# 1. The Single Crane Lift Sequence Problem

## Tier 1 — Dynamic Programming (DP)

This notebook turns a “pen & paper” DP formulation into **fully runnable code**.

### Learning goals

- Understand what a DP *state* is in sequencing problems.
- See how **mode switching** (single vs dual) changes the objective.
- Learn how to reconstruct an optimal plan from a cached DP recursion.

### What this notebook outputs

- An optimal lift plan for a small instance.
- A table with per-step time contributions.
- A visualization of the bay + the crane’s movement.

### Why the instance is small

DP enumerates many subsets, so we intentionally use a small example (6 containers) to keep the logic transparent and fast.


In [ ]:
# Environment check (no installs here)
#
# Best practice for classes: preinstall dependencies in the Docker/JupyterHub image.
# If you're running locally, install packages once in your environment.

try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
except ImportError as e:
    raise ImportError(
        "Missing dependency. Install: numpy, pandas, matplotlib. "
        "If you use the provided JupyterHub Docker image, these should already be installed."
    ) from e

print("Dependencies imported successfully.")


## Concrete instance (6 containers)

We will solve a small 2×3 bay (6 positions). Each container has:

- `id`
- `pos` (position index 1..6)
- `weight` (tons)

### Time model

- Single handling time: `2.5` minutes
- Dual handling time: `3.8` minutes
- Mode switch setup time: `1.2` minutes
- Travel time (minutes): `0.8 * |pos_from - pos_to|`

### Dual feasibility

A dual lift is feasible when:

- the two containers are adjacent on the grid
- their combined weight is ≤ the limit

We keep the model small so you can focus on *how DP works*.


In [ ]:
# ----------------------------
# Imports
# ----------------------------
# `dataclass` gives us a clean way to define small data objects.
# `lru_cache` memoizes the DP function so we don't recompute the same state.
# `combinations` helps us iterate over pairs for possible dual lifts.
from dataclasses import dataclass
from functools import lru_cache
from itertools import combinations
from typing import Literal


# ----------------------------
# Small type alias (readability)
# ----------------------------
# We only allow two strings for crane mode.
Mode = Literal["single", "dual"]


# ----------------------------
# Data model: a container
# ----------------------------
# We keep only what we need for Tier 1.
@dataclass(frozen=True)
class Container:
    # Unique identifier (1..6)
    id: int
    # Position index (1..6) in the bay
    pos: int
    # Weight in tons
    weight: float


# ----------------------------
# Concrete 6-container instance
# ----------------------------
# This is intentionally small so DP runs instantly and the logic stays readable.
containers = [
    Container(1, 1, 8.5),
    Container(2, 2, 12.3),
    Container(3, 3, 15.7),
    Container(4, 4, 9.8),
    Container(5, 5, 11.2),
    Container(6, 6, 14.1),
]

# Fast lookup: container id -> Container object
id_to_container = {c.id: c for c in containers}


# ----------------------------
# Time model parameters (minutes)
# ----------------------------
# These constants define how costly actions are.
HANDLE_SINGLE = 2.5
HANDLE_DUAL = 3.8
SETUP_TIME = 1.2
DUAL_WEIGHT_LIMIT = 20.0
TRAVEL_PER_UNIT = 0.8  # minutes per |pos_i - pos_j|

# Starting conditions
START_POS = 1.0
START_MODE: Mode = "single"


# ----------------------------
# Helper functions
# ----------------------------
def travel_time(pos_from: float, pos_to: float) -> float:
    """Compute travel time (minutes) as a simple linear function of distance."""
    return TRAVEL_PER_UNIT * abs(pos_from - pos_to)


def dual_center_pos(id1: int, id2: int) -> float:
    """Return the crane 'center position' for lifting two containers at once."""
    return (id_to_container[id1].pos + id_to_container[id2].pos) / 2.0


def dual_feasible(id1: int, id2: int) -> bool:
    """Check the weight feasibility constraint for a dual lift."""
    return (id_to_container[id1].weight + id_to_container[id2].weight) <= DUAL_WEIGHT_LIMIT


# Feasible dual pairs by *weight only* (we will also apply adjacency later).
feasible_dual_pairs = [
    pair
    for pair in combinations([c.id for c in containers], 2)
    if dual_feasible(*pair)
]

feasible_dual_pairs

## Step 1 — Define when a dual lift is allowed

We will treat the bay as a 2×3 grid with positions numbered in a **serpentine** order (a common convention when numbering along travel paths):

```text
Row 0:  1  2  3
Row 1:  6  5  4
```

A **dual** lift is allowed when:

- The two containers are **adjacent on the grid** (Manhattan distance = 1)
- Their **combined weight** is ≤ the dual weight limit

This makes it possible to have feasible pairs like (3,4) and (2,5).


In [ ]:
# ----------------------------
# Grid adjacency model (for dual lifts)
# ----------------------------
# In Tier 1 we need to know *which two positions are adjacent*.
# The bay is 2x3 and we use a serpentine numbering:
#
# Row 0:  1  2  3
# Row 1:  6  5  4
#
# This is a common numbering when a vehicle/crane path snakes through rows.
pos_to_rc = {
    1: (0, 0),
    2: (0, 1),
    3: (0, 2),
    4: (1, 2),
    5: (1, 1),
    6: (1, 0),
}


def are_adjacent_on_grid(pos_i: int, pos_j: int) -> bool:
    """Return True if positions are neighbors in the 2D grid (Manhattan distance = 1)."""
    ri, ci = pos_to_rc[pos_i]
    rj, cj = pos_to_rc[pos_j]

    # Manhattan distance on a grid: |dr| + |dc|
    return abs(ri - rj) + abs(ci - cj) == 1


# Dual lifts must satisfy BOTH constraints:
# - weight feasibility (already checked in `feasible_dual_pairs`)
# - adjacency feasibility (checked here)
adjacent_dual_pairs = []
for (i, j) in feasible_dual_pairs:
    pos_i = id_to_container[i].pos
    pos_j = id_to_container[j].pos
    if are_adjacent_on_grid(pos_i, pos_j):
        adjacent_dual_pairs.append((i, j))

adjacent_dual_pairs

## Step 2 — Dynamic programming to find the best lift plan

We define a DP state by:

- `remaining`: the set of containers that are not yet lifted
- `pos`: the crane's current position index (an integer for single lifts, or a half-integer for dual centers)
- `mode`: current crane mode (`single` or `dual`)

At each step we choose:

- A **single** lift of any remaining container
- A **dual** lift of an *adjacent* feasible pair

and accumulate:

- travel time
- mode switch time (if the chosen lift mode differs from current mode)
- handling time

We return the minimum total time and reconstruct the sequence.


In [ ]:
from typing import FrozenSet, List, Dict, Any


# ----------------------------
# Cost of taking one lift action
# ----------------------------
# This function is used by both DP and the reconstruction step.
#
# Inputs:
# - current_pos: where the crane is now
# - current_mode: single or dual
# - lift_mode: the mode we want to use for the next action
# - target_pos: where the crane must travel to perform that action
#
# Output:
# - total time for this step = travel + (optional setup) + handling

def lift_cost(current_pos: float, current_mode: Mode, lift_mode: Mode, target_pos: float) -> float:
    # Setup time is paid only if we switch modes.
    mode_switch = SETUP_TIME if current_mode != lift_mode else 0.0

    # Handling time depends on whether we lift one or two containers.
    handle = HANDLE_DUAL if lift_mode == "dual" else HANDLE_SINGLE

    # Travel time is a simple distance-based function.
    travel = travel_time(current_pos, target_pos)

    return travel + mode_switch + handle


# ----------------------------
# DP definition
# ----------------------------
# State:
# - remaining: which container IDs are still not lifted
# - pos: crane position (can be integer or half-integer for dual centers)
# - mode: current mode ('single' or 'dual')
#
# Value:
# - minimal remaining time to finish all remaining containers
#
# We cache (`lru_cache`) because the same state can be reached in many ways.
@lru_cache(maxsize=None)
def dp(remaining: FrozenSet[int], pos: float, mode: Mode) -> float:
    # Base case: nothing left to lift.
    if not remaining:
        return 0.0

    best = float("inf")

    # ----------------------------
    # Option A: choose a single lift
    # ----------------------------
    for cid in remaining:
        c = id_to_container[cid]

        # Cost to go lift container `cid` as a single lift.
        step = lift_cost(pos, mode, "single", float(c.pos))

        # After lifting it, we remove it from `remaining`, and update (pos, mode).
        remaining2 = frozenset(set(remaining) - {cid})
        best = min(best, step + dp(remaining2, float(c.pos), "single"))

    # ----------------------------
    # Option B: choose a dual lift (two containers at once)
    # ----------------------------
    rem_list = sorted(remaining)
    for i, j in combinations(rem_list, 2):
        # Skip pairs that are not allowed (not adjacent or not feasible).
        if (i, j) not in adjacent_dual_pairs and (j, i) not in adjacent_dual_pairs:
            continue

        center = dual_center_pos(i, j)
        step = lift_cost(pos, mode, "dual", center)

        # Remove both containers and update state.
        remaining2 = frozenset(set(remaining) - {i, j})
        best = min(best, step + dp(remaining2, center, "dual"))

    return best


# ----------------------------
# Reconstruct an optimal plan
# ----------------------------
# DP gives the best value (time) but not the actual actions.
# So we re-run the choices and pick the action that matches the DP optimum.

def reconstruct() -> List[Dict[str, Any]]:
    remaining: FrozenSet[int] = frozenset(c.id for c in containers)
    pos: float = float(START_POS)
    mode: Mode = START_MODE

    plan: List[Dict[str, Any]] = []

    while remaining:
        best_choice = None
        best_value = float("inf")

        # Try all single actions
        for cid in remaining:
            c = id_to_container[cid]
            step = lift_cost(pos, mode, "single", float(c.pos))
            value = step + dp(frozenset(set(remaining) - {cid}), float(c.pos), "single")
            if value < best_value:
                best_value = value
                best_choice = ("single", [cid], float(c.pos))

        # Try all dual actions
        rem_list = sorted(remaining)
        for i, j in combinations(rem_list, 2):
            if (i, j) not in adjacent_dual_pairs and (j, i) not in adjacent_dual_pairs:
                continue
            center = dual_center_pos(i, j)
            step = lift_cost(pos, mode, "dual", center)
            value = step + dp(frozenset(set(remaining) - {i, j}), center, "dual")
            if value < best_value:
                best_value = value
                best_choice = ("dual", [i, j], center)

        assert best_choice is not None

        lift_mode_str, ids, new_pos = best_choice
        lift_mode: Mode = "dual" if lift_mode_str == "dual" else "single"

        # Compute and store step time for reporting
        step_cost = lift_cost(pos, mode, lift_mode, new_pos)
        plan.append(
            {
                "mode": lift_mode,
                "containers": ids,
                "to_pos": new_pos,
                "step_time": step_cost,
            }
        )

        # Update state
        remaining = frozenset(set(remaining) - set(ids))
        pos = new_pos
        mode = lift_mode

    return plan


# ----------------------------
# Solve the instance
# ----------------------------
opt_time = dp(frozenset(c.id for c in containers), float(START_POS), START_MODE)
plan = reconstruct()

opt_time, plan

## Step 3 — Inspect and validate the optimal plan

A good habit with optimization code is to validate that the reconstructed plan matches the objective.

We will check:

- Every container is lifted exactly once.
- The sum of step times equals the DP value.

Then we will look at the plan in a table.


In [ ]:
# ----------------------------
# Convert plan (list of dicts) into a clean table
# ----------------------------
# Beginners often find it easier to understand results in a table.

# Build a DataFrame from the reconstructed plan.
df = pd.DataFrame(plan)

# Make row numbers start at 1 (more natural for step-by-step schedules).
df.index = np.arange(1, len(df) + 1)

# Convert list of container IDs into a readable string like "2,5".
df["containers"] = df["containers"].apply(lambda xs: ",".join(map(str, xs)))

# Round times so the output is easy to read.
df["step_time"] = df["step_time"].round(3)

# Sanity checks
# 1) Sum of step times should match the DP optimum.
displayed_total = float(df["step_time"].sum())

# 2) Each container should appear exactly once.
all_lifted = sorted([int(x) for step in plan for x in step["containers"]])

print(f"DP optimal total time: {opt_time:.3f} min")
print(f"Sum of reconstructed step times: {displayed_total:.3f} min")
print("Lifted container IDs:", all_lifted)
print("All containers lifted exactly once:", all_lifted == sorted([c.id for c in containers]))

# Show the table
# Note: in Jupyter, the last expression is displayed.
df

## Additional operational visualizations (logistics-focused)

Besides the bay/path plot, it is useful to visualize:

- **time breakdown per step** (travel vs setup vs handling)
- **cumulative completion time** (how total time grows step-by-step)
- a simple **timeline / Gantt-like view** of operations

These plots are derived from the reconstructed plan.

In [ ]:
# ----------------------------
# Operational visualizations from the DP plan
# ----------------------------
# For Tier 1 we currently have only `step_time` per step.
# We can still visualize:
# - step time bars
# - cumulative completion time
# - a Gantt-like timeline (sequence over time)

vis = df.copy()

# Make a numeric step index (1..n)
vis["step"] = vis.index.astype(int)

# Cumulative time after each step
vis["cum_time"] = vis["step_time"].cumsum()

# Start time for each step = cumulative - duration
vis["start_time"] = vis["cum_time"] - vis["step_time"]

# Color by lift mode
color_map = {"single": "#2E90FA", "dual": "#12B76A"}
colors = [color_map[m] for m in vis["mode"].tolist()]

fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))

# (1) Step durations
axes[0].bar(vis["step"], vis["step_time"], color=colors, edgecolor="#101828", alpha=0.85)
axes[0].set_title("Step durations")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Minutes")
axes[0].grid(True, axis="y", alpha=0.25)

# (2) Cumulative completion time
axes[1].plot(vis["step"], vis["cum_time"], marker="o", lw=2, color="#344054")
axes[1].set_title("Cumulative time")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Minutes")
axes[1].grid(True, alpha=0.25)

# (3) Timeline / Gantt-like view
axes[2].barh(
    y=vis["step"],
    width=vis["step_time"],
    left=vis["start_time"],
    color=colors,
    edgecolor="#101828",
    alpha=0.85,
)
axes[2].set_title("Timeline (Gantt-like)")
axes[2].set_xlabel("Minutes")
axes[2].set_ylabel("Step")
axes[2].grid(True, axis="x", alpha=0.25)

# Legend
from matplotlib.patches import Patch

legend_handles = [
    Patch(facecolor=color_map["single"], edgecolor="#101828", label="single"),
    Patch(facecolor=color_map["dual"], edgecolor="#101828", label="dual"),
]
axes[2].legend(handles=legend_handles, loc="best")

plt.tight_layout()
plt.show()

## Step 4 — Visualization and interpretation

The visualization below helps you understand *why* a plan is good.

- Shorter arrows usually mean lower travel time.
- Dual steps can remove two containers at once, but may require a mode switch.

After viewing the plot:

- Compare it with the table to see which moves are costly.
- Ask: "Where does the DP choose to pay setup time, and why?"


In [ ]:
# ----------------------------
# Visualization: bay layout + crane movement
# ----------------------------
# Goal: help you *see* the difference between long travel moves and short ones,
# and where dual lifts happen.

# Convert each position (1..6) into x,y coordinates for plotting.
# - x is column
# - y is row (we use -row so row 0 appears on top)
pos_to_xy = {p: (c, -r) for p, (r, c) in pos_to_rc.items()}

fig, ax = plt.subplots(figsize=(6, 3.5))

# ----------------------------
# Draw the 6 grid cells
# ----------------------------
for p, (x, y) in pos_to_xy.items():
    ax.scatter(
        [x],
        [y],
        s=800,
        marker="s",
        facecolor="#F2F4F7",
        edgecolor="#344054",
    )
    ax.text(
        x,
        y,
        str(p),
        ha="center",
        va="center",
        fontsize=12,
        color="#101828",
    )

# ----------------------------
# Label containers in each cell
# ----------------------------
# We show container id and weight.
for c in containers:
    x, y = pos_to_xy[c.pos]
    ax.text(
        x,
        y - 0.25,
        f"id={c.id}\nw={c.weight}",
        ha="center",
        va="top",
        fontsize=9,
        color="#475467",
    )

# ----------------------------
# Build the crane path as a list of points
# ----------------------------
# We start at START_POS, then visit each step's target position.
path_xy = []

current_pos = float(START_POS)
# Starting point (slightly above the cell so arrows are visible)
x0, y0 = pos_to_xy[int(round(current_pos))]
path_xy.append((x0, y0 + 0.35))

for step in plan:
    to_pos = float(step["to_pos"])

    if to_pos.is_integer():
        # Single lift: target is exactly a cell position.
        x, y = pos_to_xy[int(to_pos)]
    else:
        # Dual lift: target is a "center" between two adjacent cells.
        left = int(np.floor(to_pos))
        right = int(np.ceil(to_pos))
        x1, y1 = pos_to_xy[left]
        x2, y2 = pos_to_xy[right]
        x, y = (x1 + x2) / 2, (y1 + y2) / 2

    path_xy.append((x, y + 0.35))

# ----------------------------
# Draw arrows between consecutive path points
# ----------------------------
for (x1, y1), (x2, y2) in zip(path_xy[:-1], path_xy[1:]):
    ax.annotate(
        "",
        xy=(x2, y2),
        xytext=(x1, y1),
        arrowprops=dict(arrowstyle="->", lw=2, color="#2E90FA"),
    )

# Final plot formatting
ax.set_title("Tier 1 DP — Bay layout and crane path", fontsize=12)
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlim(-0.75, 2.75)
ax.set_ylim(-1.75, 0.75)
ax.set_aspect("equal")

plt.show()

## Common pitfalls (and how to debug)

- If dual lifts never appear:
  - Check adjacency rules
  - Check weight limits
- If total time looks wrong:
  - Print per-step travel/setup/handle components
  - Verify that position updates are correct for dual centers

## Next steps

- Try changing the weights and the dual weight limit.
- Increase the number of containers and compare:
  - DP runtime (grows fast)
  - heuristic runtime (stays fast)


## Sensitivity (pedagogical add-on): when do dual lifts appear?

In the concrete instance above, it is possible that **no dual pair is feasible** under the current weight limit + adjacency rule. That can hide the key idea of Tier 1 (mode switching and dual actions).

So we run a small *what-if* experiment:

- keep the same bay layout + weights
- vary `DUAL_WEIGHT_LIMIT`
- recompute feasible adjacent dual pairs
- re-run DP and summarize whether the optimal plan uses dual steps

This helps you *see* how constraints control the solution space.

In [ ]:
# What-if: vary the dual weight limit and see whether DP ever uses dual lifts.
#
# This section is intentionally compact: it reuses dp() but recomputes adjacency-feasible pairs.
# It does NOT change the main example; it just adds an educational experiment.


def solve_with_dual_limit(limit: float) -> Dict[str, Any]:
    global DUAL_WEIGHT_LIMIT, feasible_dual_pairs, adjacent_dual_pairs

    # Save current globals so we can restore them after the experiment.
    old_limit = float(DUAL_WEIGHT_LIMIT)
    old_feasible = feasible_dual_pairs.copy()
    old_adjacent = adjacent_dual_pairs.copy()

    # Temporarily override the global limit
    DUAL_WEIGHT_LIMIT = float(limit)

    # Recompute feasible pairs by weight
    feasible_dual_pairs = [
        pair
        for pair in combinations([c.id for c in containers], 2)
        if dual_feasible(*pair)
    ]

    # Recompute adjacency-feasible dual pairs
    adjacent_dual_pairs = []
    for (i, j) in feasible_dual_pairs:
        pos_i = id_to_container[i].pos
        pos_j = id_to_container[j].pos
        if are_adjacent_on_grid(pos_i, pos_j):
            adjacent_dual_pairs.append((i, j))

    # Capture experiment-specific values BEFORE restoring globals
    exp_adjacent_pairs = adjacent_dual_pairs.copy()

    # Clear DP cache because the transition set changed
    dp.cache_clear()

    opt_time2 = dp(frozenset(c.id for c in containers), float(START_POS), START_MODE)
    plan2 = reconstruct()

    dual_steps = sum(1 for step in plan2 if step["mode"] == "dual")

    # Restore original globals
    DUAL_WEIGHT_LIMIT = old_limit
    feasible_dual_pairs = old_feasible
    adjacent_dual_pairs = old_adjacent
    dp.cache_clear()

    return {
        "dual_weight_limit": float(limit),
        "num_adjacent_dual_pairs": int(len(exp_adjacent_pairs)),
        "adjacent_dual_pairs": exp_adjacent_pairs,
        "opt_time": float(opt_time2),
        "dual_steps": int(dual_steps),
    }


limits_to_try: List[float] = [20.0, 22.0, 24.0, 26.0, 30.0]
rows = [solve_with_dual_limit(L) for L in limits_to_try]

pd.DataFrame(rows)